# IBM Coursera — Python Project for Data Science
## Tesla and GameStop stock/revenue analysis

This notebook organizes the seven items in the peer-graded final project. Run the cells in order. For Coursera, upload your **own screenshots of the executed code and outputs**; do not submit blank/unexecuted output as if it were run.

**Data note:** financial websites can change their HTML or block automated requests. If the revenue-scraping cell fails, use the revenue dataframe from your completed Coursera web-scraping lab and continue with the `tail()` and graph cells.

## Setup

In [ ]:
%pip install -q yfinance pandas matplotlib requests beautifulsoup4 lxml

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
from io import StringIO


## Question 1 — Extract Tesla stock data using yfinance

Create the `tesla` ticker, download the maximum available history, reset the index, and display the first five rows.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2 — Extract Tesla revenue data using web scraping

This cell uses the IBM course-provided Tesla revenue page. It selects the column explicitly labeled Revenue so the year is never mistaken for the revenue amount.

In [ ]:
# Question 2 — Use Webscraping to Extract Tesla Revenue Data
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text

# The IBM course instructions identify the Tesla revenue table as table index 1.
tesla_revenue = pd.read_html(StringIO(html_data))[1]
tesla_revenue.columns = ["Date", "Revenue"]
tesla_revenue["Revenue"] = (
    tesla_revenue["Revenue"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
)
tesla_revenue["Revenue"] = pd.to_numeric(tesla_revenue["Revenue"], errors="coerce")
tesla_revenue = tesla_revenue.dropna(subset=["Revenue"]).reset_index(drop=True)
tesla_revenue.tail()

## Question 3 — Extract GameStop stock data using yfinance

Create the `GME` ticker, download the maximum available history, reset the index, and display the first five rows.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4 — Extract GameStop revenue data using web scraping

This cell uses the IBM course-provided GameStop revenue page and selects the column explicitly labeled Revenue.

In [ ]:
# Question 4 — Use Webscraping to Extract GameStop Revenue Data
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data = requests.get(url).text

# The IBM course instructions identify the GameStop revenue table as table index 1.
gme_revenue = pd.read_html(StringIO(html_data))[1]
gme_revenue.columns = ["Date", "Revenue"]
gme_revenue["Revenue"] = (
    gme_revenue["Revenue"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
)
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"], errors="coerce")
gme_revenue = gme_revenue.dropna(subset=["Revenue"]).reset_index(drop=True)
gme_revenue.tail()

## Graph helper

The course asks you to use a `make_graph` function. This implementation plots the stock closing price and overlays revenue on a second y-axis when a revenue dataframe is provided.

In [ ]:
def make_graph(stock_data, revenue_data, stock_name):
    stock = stock_data.copy()
    if "Date" not in stock.columns:
        stock = stock.reset_index()
    stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce")
    stock = stock.dropna(subset=["Date"]).sort_values("Date")

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.plot(stock["Date"], stock["Close"], label="Stock closing price")
    ax1.set_xlabel("Date")
    ax1.set_ylabel("Stock closing price (USD)")
    ax1.set_title(f"{stock_name} Stock Price and Revenue")
    ax1.grid(True, alpha=0.3)

    if revenue_data is not None and not revenue_data.empty:
        rev = revenue_data.copy()
        # Normalize common column names across scraped tables.
        rev.columns = [str(c).strip() for c in rev.columns]
        date_col = next((c for c in rev.columns if "date" in c.lower()), None)
        revenue_col = next((c for c in rev.columns if "revenue" in c.lower()), None)
        if date_col and revenue_col:
            rev[date_col] = pd.to_datetime(rev[date_col], errors="coerce")
            rev[revenue_col] = (
                rev[revenue_col].astype(str).str.replace(r"[$,]", "", regex=True)
            )
            rev[revenue_col] = pd.to_numeric(rev[revenue_col], errors="coerce")
            rev = rev.dropna(subset=[date_col, revenue_col]).sort_values(date_col)
            if not rev.empty:
                ax2 = ax1.twinx()
                ax2.plot(rev[date_col], rev[revenue_col], linestyle="--",
                         label="Revenue", alpha=0.8)
                ax2.set_ylabel("Revenue")
                lines1, labels1 = ax1.get_legend_handles_labels()
                lines2, labels2 = ax2.get_legend_handles_labels()
                ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
    plt.tight_layout()
    plt.show()

## Question 5 — Plot Tesla stock graph

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 — Plot GameStop stock graph

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Question 7 — Share your assignment notebook

1. Save this notebook as `Final_Project_Tesla_GameStop.ipynb`.
2. Upload it to a GitHub repository you own.
3. Paste the notebook's GitHub URL into the Coursera URL field.
4. Upload screenshots for Questions 1–6 showing the question/code and the actual output from your executed notebook.

**Before submitting:** verify that both revenue dataframes contain real rows and that both graphs display correctly. A failed scrape or empty revenue dataframe is not a completed result.